# WP3 Africa Hazard Ranking — Dataset Notebook 05 
## IDMC (GIDD disasters) — disaster internal displacement extraction (country × hazard)

### What this notebook produces
A cleaned, hazard-mapped, **country × hazard** extraction of **IDMC disaster internal displacements**, and an upsert into:

`data/intermediate/wp3_country_indicators_long.csv`

### Key transparency notes
- **IDMC disaster displacement coverage:** this file covers **2008–2024** (the notebook checks and prints this explicitly).
  The WP3 framework may refer to 2000–2024, but IDMC’s disaster series is generally available from **2008 onwards**, so we use the
  intersection of [2000–2024] with available years and record the *true* time window in `time_window`.
- **Hazard mapping:** uses **Hazard Category / Type / Sub Type**, and enhances flood/storm categorisation using:
  - **Event Codes prefixes** where present (e.g., `FF` for flash flood, `TC` for tropical cyclone), and
  - **Event Name regex** (keywords) as a fallback / disambiguation step.
  Every row stores `mapping_rule` and `mapping_confidence`. Unmapped cases are exported for review.
- **Population denominator:** uses a **World Bank population CSV** provided (`POP(POP).csv`).
  This file is a **global “Population table** (ISO3 + population in thousands):
  - uses it as a temporary denominator,
  - stores `pop_ref_year=YYYY`, and
  - states the year explicitly in the indicator `notes`.

---

## Inputs (expected paths)
- Scope list: `data/raw/scope/ISO3_Regions.csv`
- IDMC file(s): `data/raw/IDMC/*.xlsx`
- Population: `data/raw/population/*.zip`

---

## Outputs
- `data/intermediate/idmc_mapped_events_2000_2024.csv` (record-level with mapping + audit fields)
- `data/intermediate/idmc_country_hazard_agg_2000_2024.csv` (country×hazard aggregates)
- `data/intermediate/idmc_mapping_audit_unmapped.csv` (unmapped combos)
- `data/intermediate/idmc_population_ref_from_worldbank.csv` (population reference table)
- Upserted master: `data/intermediate/wp3_country_indicators_long.csv`

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from hashlib import md5
import zipfile

# -----------------------------
# Config — edit paths if needed
# -----------------------------
PROJECT_ROOT = Path(".").resolve()

# Inputs (repo-relative)
COUNTRY_SCOPE_PATH = PROJECT_ROOT.parent / "data" / "raw" / "scope" / "ISO3_Regions.csv"
IDMC_DIR           = PROJECT_ROOT.parent / "data" / "raw" / "IDMC"
POP_DIR            = PROJECT_ROOT.parent / "data" / "raw" / "population"
WB_POP_PATH        = POP_DIR / "API_SP.POP.TOTL_DS2_en_csv_v2_34.zip"

# Sandbox fallbacks
FALLBACK_SCOPE = Path("/mnt/data/ISO3_Regions.csv")
FALLBACK_IDMC  = Path("/mnt/data/IDMC_GIDD_Disasters_Internal_Displacement_Data (1).xlsx")
FALLBACK_WDI_ZIP = Path("/mnt/data/API_SP.POP.TOTL_DS2_en_csv_v2_34.zip")

if not COUNTRY_SCOPE_PATH.exists() and FALLBACK_SCOPE.exists():
    COUNTRY_SCOPE_PATH = FALLBACK_SCOPE

# Master output
MASTER_PATH = PROJECT_ROOT.parent / "data" / "intermediate" / "wp3_country_indicators_long.csv"
MASTER_PATH.parent.mkdir(parents=True, exist_ok=True)

# Outputs
OUT_DIR = PROJECT_ROOT.parent / "data" / "intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_MAPPED     = OUT_DIR / "idmc_mapped_events_2000_2024.csv"
OUT_AGG        = OUT_DIR / "idmc_country_hazard_agg_2000_2024.csv"
OUT_UNMAPPED   = OUT_DIR / "idmc_mapping_audit_unmapped.csv"
OUT_POP_REF    = OUT_DIR / "idmc_population_ref_from_worldbank.csv"

print("PROJECT_ROOT:", PROJECT_ROOT.parent)
print("COUNTRY_SCOPE_PATH:", COUNTRY_SCOPE_PATH, "exists:", COUNTRY_SCOPE_PATH.exists())
print("IDMC_DIR:", IDMC_DIR, "exists:", IDMC_DIR.exists())
print("WB_POP_PATH:", WB_POP_PATH, "exists:", WB_POP_PATH.exists())
print("MASTER_PATH:", MASTER_PATH)

PROJECT_ROOT: C:\pipelines\sewa-multihazar
COUNTRY_SCOPE_PATH: C:\pipelines\sewa-multihazar\data\raw\scope\ISO3_Regions.csv exists: True
IDMC_DIR: C:\pipelines\sewa-multihazar\data\raw\IDMC exists: True
WB_POP_PATH: C:\pipelines\sewa-multihazar\data\raw\population\API_SP.POP.TOTL_DS2_en_csv_v2_34.zip exists: True
MASTER_PATH: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv


## Global setup — long-format contract and upsert

In [3]:
HAZARDS_9 = [
    "Drought",
    "Flash Flooding",
    "Riverine Flooding Continued & Cluster",
    "Heatwave",
    "Storm Surge",
    "Wind",
    "Thunderstorm",
    "Wildfires",
    "Dust",
]
DIMENSIONS_5 = ["Prevalence", "Scale", "Impact", "Cascade impacts", "Future relevance"]
SOURCE = "IDMC (GIDD disasters)"

LONG_COLUMNS = [
    "iso3","country_name","region","hazard","dimension","source",
    "indicator_id","indicator_name","value_raw","unit_raw",
    "year","time_window","notes"
]
UPSERT_KEY = ["iso3","hazard","dimension","source","indicator_id","year"]

def assert_contract(df: pd.DataFrame) -> None:
    missing_cols = [c for c in LONG_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Contract violation: missing columns {missing_cols}")

    bad_dims = sorted(set(df["dimension"]) - set(DIMENSIONS_5))
    if bad_dims:
        raise ValueError(f"Unknown dimensions {bad_dims}")

    allowed_haz = set(HAZARDS_9)
    bad_haz = sorted(set(df["hazard"]) - allowed_haz)
    if bad_haz:
        raise ValueError(f"Unknown hazards {bad_haz}")

    dup = df.duplicated(UPSERT_KEY, keep=False)
    if dup.any():
        raise ValueError("Duplicate UPSERT keys found. Check indicator_id/source/year logic.")

def upsert_to_master(new_df: pd.DataFrame, master_path: Path) -> pd.DataFrame:
    assert_contract(new_df)

    if master_path.exists():
        master = pd.read_csv(master_path)
        for col in LONG_COLUMNS:
            if col not in master.columns:
                master[col] = np.nan
        master = master[LONG_COLUMNS]

        master_key = master[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        new_key    = new_df[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        master = master.loc[~master_key.isin(set(new_key))].copy()

        out = pd.concat([master, new_df[LONG_COLUMNS]], ignore_index=True)
    else:
        out = new_df[LONG_COLUMNS].copy()

    out.to_csv(master_path, index=False)
    return out

## Step 1 — Load scope list (ISO3 + region)

In [4]:
scope_raw = pd.read_csv(COUNTRY_SCOPE_PATH)
scope_raw.columns = [c.replace("\xa0"," ").strip() for c in scope_raw.columns]

scope = scope_raw.copy()
scope["Region"] = scope["Region"].replace({np.nan: None}).ffill()
scope["Region"] = scope["Region"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["Country"] = scope["Country"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["ISO3.Code"] = scope["ISO3.Code"].astype(str).str.strip().str.upper()

scope = scope.rename(columns={"Region":"region","Country":"country_name","ISO3.Code":"iso3"})
scope = scope[["iso3","country_name","region"]].drop_duplicates()

if scope["iso3"].duplicated().any():
    raise ValueError("Duplicate ISO3 in scope list.")

print("Scope countries:", len(scope))
scope.head()

Scope countries: 47


,iso3,country_name,region
0,NGA,Nigeria,West Africa
1,GHA,Ghana,West Africa
2,SEN,Senegal,West Africa
3,BFA,Burkina Faso,West Africa
4,MLI,Mali,West Africa


## Step 2 — Load IDMC disasters export

In [5]:
# Find IDMC excel in repo dir
idmc_candidates = []
if IDMC_DIR.exists():
    idmc_candidates = sorted(list(IDMC_DIR.glob("*.xlsx")))
if not idmc_candidates and FALLBACK_IDMC.exists():
    idmc_candidates = [FALLBACK_IDMC]

if not idmc_candidates:
    raise FileNotFoundError("No IDMC .xlsx found in data/raw/IDMC and no fallback found.")

IDMC_PATH = idmc_candidates[0]
print("Using IDMC file:", IDMC_PATH)

xls = pd.ExcelFile(IDMC_PATH)
sheet = "1_Disaster_Displacement_data" if "1_Disaster_Displacement_data" in xls.sheet_names else xls.sheet_names[0]
idmc = pd.read_excel(IDMC_PATH, sheet_name=sheet)

print("Rows:", len(idmc), "Cols:", len(idmc.columns))
idmc.head()

Using IDMC file: C:\pipelines\sewa-multihazar\data\raw\IDMC\IDMC_GIDD_Disasters_Internal_Displacement_Data (1).xlsx
Rows: 22119 Cols: 13


,ISO3,Country / Territory,Year,Event Name,Date of Event (start),Disaster Internal Displacements,Disaster Internal Displacements (Raw),Hazard Category,Hazard Type,Hazard Sub Type,Event Codes (Code:Type),Event ID,Displacement occurred
0,AFG,Afghanistan,2008,Afghanistan: Flood - 01/08/2008,2008-08-01,180,180,Weather related,Flood,Flood,NaN,NaN,NaN
1,AFG,Afghanistan,2008,Afghanistan: Earthquake - 17/04/2008,2008-04-17,3200,3250,Geophysical,Earthquake,Earthquake,NaN,NaN,NaN
2,ARG,Argentina,2008,Argentina: Flood - 28/01/2008,2008-01-28,640,635,Weather related,Flood,Flood,NaN,NaN,NaN
3,ATG,Antigua and Barbuda,2008,Antigua and Barbuda: Storm - 15/10/2008,2008-10-15,45,45,Weather related,Storm,Storm,NaN,NaN,NaN
4,AUS,Australia,2008,Australia: Storm - 16/11/2008,2008-11-16,1000,1000,Weather related,Storm,Storm,NaN,NaN,NaN


## Step 3 — Basic cleaning + framework year filter (2008–2024)

In [6]:
idmc.columns = [c.strip() for c in idmc.columns]

required = [
    "ISO3","Country / Territory","Year","Event Name",
    "Date of Event (start)",
    "Disaster Internal Displacements",
    "Disaster Internal Displacements (Raw)",
    "Hazard Category","Hazard Type","Hazard Sub Type",
    "Event Codes (Code:Type)"
]
missing = [c for c in required if c not in idmc.columns]
if missing:
    raise ValueError(f"IDMC file missing required columns: {missing}")

idmc["ISO3"] = idmc["ISO3"].astype(str).str.strip().str.upper()
idmc["Country / Territory"] = idmc["Country / Territory"].astype(str).str.strip()
idmc["Event Name"] = idmc["Event Name"].astype(str).str.strip()

idmc["Year"] = pd.to_numeric(idmc["Year"], errors="coerce").astype("Int64")
idmc["Disaster Internal Displacements (Raw)"] = pd.to_numeric(idmc["Disaster Internal Displacements (Raw)"], errors="coerce")
idmc["Disaster Internal Displacements"] = pd.to_numeric(idmc["Disaster Internal Displacements"], errors="coerce")

# Prefer raw values when available
idmc["displacements_raw_preferred"] = idmc["Disaster Internal Displacements (Raw)"].fillna(idmc["Disaster Internal Displacements"])

# Parse start date (best effort)
idmc["start_date"] = pd.to_datetime(idmc["Date of Event (start)"], errors="coerce")

YEAR_MIN_FRAMEWORK, YEAR_MAX_FRAMEWORK = 2000, 2024
yr_min_data = int(idmc["Year"].min())
yr_max_data = int(idmc["Year"].max())
print("IDMC year coverage in file:", yr_min_data, "to", yr_max_data)

idmc = idmc.loc[(idmc["Year"] >= YEAR_MIN_FRAMEWORK) & (idmc["Year"] <= YEAR_MAX_FRAMEWORK)].copy()
print("Rows within 2000–2024 filter:", len(idmc))

# Filter to WP3 scope countries early
idmc = idmc.merge(scope, left_on="ISO3", right_on="iso3", how="inner")
print("Rows after scope filter:", len(idmc))

idmc.head()

IDMC year coverage in file: 2008 to 2024
Rows within 2000–2024 filter: 22119
Rows after scope filter: 3537


,ISO3,Country / Territory,Year,Event Name,Date of Event (start),Disaster Internal Displacements,Disaster Internal Displacements (Raw),Hazard Category,Hazard Type,Hazard Sub Type,Event Codes (Code:Type),Event ID,Displacement occurred,displacements_raw_preferred,start_date,iso3,country_name,region
0,BDI,Burundi,2008,Burundi: Flood - 20/09/2008,2008-09-20,2800,2770,Weather related,Flood,Flood,NaN,NaN,NaN,2770,2008-09-20,BDI,Burundi,East Africa
1,BEN,Benin,2008,Benin: Flood - 01/07/2008,2008-07-01,150000,150000,Weather related,Flood,Flood,NaN,NaN,NaN,150000,2008-07-01,BEN,Benin,West Africa
2,BFA,Burkina Faso,2008,Burkina Faso: Flood - 01/08/2008,2008-08-01,28000,28000,Weather related,Flood,Flood,NaN,NaN,NaN,28000,2008-08-01,BFA,Burkina Faso,West Africa
3,CAF,Central African Republic,2008,Central African Republic: Flood - 01/07/2008,2008-07-01,14000,14000,Weather related,Flood,Flood,NaN,NaN,NaN,14000,2008-07-01,CAF,Central African Rep.,Central Africa
4,CMR,Cameroon,2008,Cameroon: Flood - 01/07/2008,2008-07-01,1000,1000,Weather related,Flood,Flood,NaN,NaN,NaN,1000,2008-07-01,CMR,Cameroon,Central Africa


## Step 4 — Final hazard mapping (hazard fields + Event Codes + Event Name)

In [7]:
def _norm(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).strip().lower())

def extract_event_code_prefixes(s):
    if pd.isna(s):
        return set()
    prefs = set()
    for part in str(s).split(";"):
        part = part.strip()
        if not part:
            continue
        token = part.split(":")[0].strip()
        pref = token.split("-")[0].strip()
        if pref:
            prefs.add(pref.upper())
    return prefs

idmc["event_code_prefixes"] = idmc["Event Codes (Code:Type)"].apply(extract_event_code_prefixes)

# Keyword patterns
RE_FLASH = re.compile(r"\bflash\b|\bpluvial\b|\burban\b|\bcloudburst\b|\bcrue\s+eclair\b|\béclair\b", re.I)
RE_RIVER = re.compile(r"\briver\b|\boverflow\b|\bembankment\b|\blevee\b|\bdyke\b|\bdike\b|\bdam\b|\breservoir\b|\bubangi\b|\bnile\b", re.I)
RE_DAM   = re.compile(r"\bdam\b|\breservoir\b|\bspillway\b|\bdam\s+release\b|\bbreach\b|\bbroken\s+dam\b", re.I)

RE_SURGE = re.compile(r"\bstorm surge\b|\bsurge\b|\bhigh tide(s)?\b|\bking tide(s)?\b|\bcoastal flood(ing)?\b|\bsea water\b|\bseawater\b", re.I)

RE_TROPICAL = re.compile(r"\bcyclone\b|\btyphoon\b|\bhurricane\b|\btropical storm\b|\btropical cyclone\b|\btropical depression\b|\bdepression tropicale\b|\bciclone\b", re.I)
RE_WIND     = re.compile(r"\bstrong winds?\b|\bwindstorm\b|\bgale\b|\bsquall\b|\bviolent winds?\b", re.I)

RE_CONVECTIVE = re.compile(r"\brainstorm\b|\bthunderstorm\b|\blightning\b|\bhail\b|\bhailstorm\b|\btornado\b|\btwister\b|\bsevere storm\b|\btorrential rains?\b|\bheavy storm\b", re.I)

RE_DUST = re.compile(r"\bdust\b|\bsandstorm\b|\bsand storm\b|\bharmattan\b", re.I)
RE_FIRE = re.compile(r"\bwild\s*fire\b|\bwildfire\b|\bforest fire\b|\bbushfire\b|\bbrush fire\b|\bfires?\b", re.I)
RE_DROUGHT = re.compile(r"\bdrought\b|\bdry spell\b|\bwater shortage\b|\bsécheresse\b|\bsequ[ií]a\b", re.I)

def map_wp3_hazard(row):
    # Priority: hazard type/subtype -> event codes -> event name regex
    htyp = _norm(row.get("Hazard Type"))
    hsub = _norm(row.get("Hazard Sub Type"))
    evn  = str(row.get("Event Name", "") or "")
    prefs = row.get("event_code_prefixes", set()) or set()

    # Drought
    if htyp == "drought" or "DR" in prefs or RE_DROUGHT.search(evn):
        return ("Drought", "type_or_code_or_event_drought", "high" if (htyp=="drought" or "DR" in prefs) else "medium")

    # Wildfires
    if htyp == "wildfire" or "WF" in prefs or RE_FIRE.search(evn):
        return ("Wildfires", "type_or_code_or_event_fire", "high" if (htyp=="wildfire" or "WF" in prefs) else "medium")

    # Heatwave
    if htyp == "extreme temperature" and hsub == "heat wave":
        return ("Heatwave", "subtype_exact_heat_wave", "high")

    # Dust
    if (htyp == "storm" and hsub == "sand/dust storm") or RE_DUST.search(evn):
        return ("Dust", "subtype_exact_or_event_dust", "high" if (htyp=="storm" and hsub=="sand/dust storm") else "medium")

    # Storm surge
    if (htyp == "storm" and hsub == "storm surge") or RE_SURGE.search(evn):
        return ("Storm Surge", "subtype_exact_or_event_surge", "high" if (htyp=="storm" and hsub=="storm surge") else "medium")

    # Flood family
    if htyp == "flood":
        if "FF" in prefs:
            return ("Flash Flooding", "event_code_prefix_FF", "high")

        if hsub == "dam release flood" or RE_DAM.search(evn):
            return ("Riverine Flooding Continued & Cluster", "subtype_or_event_dam_release", "high" if hsub=="dam release flood" else "medium")

        if RE_SURGE.search(evn):
            return ("Storm Surge", "event_regex_surge_in_flood", "low")

        if RE_FLASH.search(evn):
            return ("Flash Flooding", "event_regex_flash", "medium")
        if RE_RIVER.search(evn):
            return ("Riverine Flooding Continued & Cluster", "event_regex_river", "medium")

        return ("Riverine Flooding Continued & Cluster", "type_fallback_generic_flood", "low")

    # Storm family
    if htyp == "storm":
        if hsub == "typhoon/hurricane/cyclone" or "TC" in prefs or RE_TROPICAL.search(evn):
            return ("Wind", "subtype_or_code_or_event_tropical", "high" if (hsub=="typhoon/hurricane/cyclone" or "TC" in prefs) else "medium")

        if hsub in {"tornado", "hailstorm"}:
            return ("Thunderstorm", "subtype_exact_convective", "high")

        if RE_WIND.search(evn):
            return ("Wind", "event_regex_wind", "medium")

        if RE_CONVECTIVE.search(evn):
            return ("Thunderstorm", "event_regex_convective", "medium")

        return ("Thunderstorm", "type_fallback_generic_storm", "low")

    # Coastal proxy
    if htyp in {"wave action", "sea level rise"}:
        return ("Storm Surge", "type_coastal_proxy", "low")

    return (None, "unmapped", "low")

idmc["wp3_hazard"], idmc["mapping_rule"], idmc["mapping_confidence"] = zip(*idmc.apply(map_wp3_hazard, axis=1))

print("Mapped rows:", idmc["wp3_hazard"].notna().sum(), "of", len(idmc))
idmc["wp3_hazard"].value_counts(dropna=False).head(20)

Mapped rows: 3348 of 3537


wp3_hazard
Riverine Flooding Continued & Cluster    1945
Thunderstorm                              757
Drought                                   305
None                                      189
Wind                                      176
Flash Flooding                            101
Wildfires                                  55
Storm Surge                                 9
Name: count, dtype: int64

## Step 5 — Event UID + record-level export for QA

In [8]:
def make_event_uid(row):
    # Prefer IDMC Event ID if present
    event_id = row.get("Event ID", None)
    if pd.notna(event_id):
        return f"IDMC_EVENTID:{str(event_id).strip()}"

    parts = [
        str(row.get("ISO3","")).strip().upper(),
        str(row.get("Year","")).strip(),
        str(row.get("Date of Event (start)","")).strip(),
        str(row.get("Hazard Type","")).strip().lower(),
        str(row.get("Hazard Sub Type","")).strip().lower(),
        str(row.get("Event Name","")).strip().lower(),
    ]
    blob = "||".join(parts)
    return "HASH:" + md5(blob.encode("utf-8")).hexdigest()

idmc["event_uid"] = idmc.apply(make_event_uid, axis=1)

mapped = idmc.loc[idmc["wp3_hazard"].notna()].copy()

mapped_out = mapped[[
    "ISO3","country_name","region","Year","start_date","Event Name",
    "Hazard Category","Hazard Type","Hazard Sub Type","Event Codes (Code:Type)",
    "event_code_prefixes",
    "wp3_hazard","mapping_rule","mapping_confidence",
    "displacements_raw_preferred","event_uid"
]].copy()

mapped_out.to_csv(OUT_MAPPED, index=False)
print("Wrote mapped events:", OUT_MAPPED, "rows:", len(mapped_out))
mapped_out.head()

Wrote mapped events: C:\pipelines\sewa-multihazar\data\intermediate\idmc_mapped_events_2000_2024.csv rows: 3348


,ISO3,country_name,region,Year,start_date,Event Name,Hazard Category,Hazard Type,Hazard Sub Type,Event Codes (Code:Type),event_code_prefixes,wp3_hazard,mapping_rule,mapping_confidence,displacements_raw_preferred,event_uid
0,BDI,Burundi,East Africa,2008,2008-09-20,Burundi: Flood - 20/09/2008,Weather related,Flood,Flood,NaN,{},Riverine Flooding Continued & Cluster,type_fallback_generic_flood,low,2770,HASH:5ccf6f8c51a340736e700d8c308f02b8
1,BEN,Benin,West Africa,2008,2008-07-01,Benin: Flood - 01/07/2008,Weather related,Flood,Flood,NaN,{},Riverine Flooding Continued & Cluster,type_fallback_generic_flood,low,150000,HASH:01db113c24ad334a2751b293c316ac32
2,BFA,Burkina Faso,West Africa,2008,2008-08-01,Burkina Faso: Flood - 01/08/2008,Weather related,Flood,Flood,NaN,{},Riverine Flooding Continued & Cluster,type_fallback_generic_flood,low,28000,HASH:40059978d4b17e7dbc229bbcd1e950b9
3,CAF,Central African Rep.,Central Africa,2008,2008-07-01,Central African Republic: Flood - 01/07/2008,Weather related,Flood,Flood,NaN,{},Riverine Flooding Continued & Cluster,type_fallback_generic_flood,low,14000,HASH:c5118e108f943378b458b6dde483f081
4,CMR,Cameroon,Central Africa,2008,2008-07-01,Cameroon: Flood - 01/07/2008,Weather related,Flood,Flood,NaN,{},Riverine Flooding Continued & Cluster,type_fallback_generic_flood,low,1000,HASH:5725257bc14838c5e835d278a67cc541


## Step 6 — Aggregate to country × hazard

In [9]:
year_min_actual = int(mapped["Year"].min())
year_max_actual = int(mapped["Year"].max())
TIME_WINDOW = f"{year_min_actual}-{year_max_actual}"

agg = (
    mapped
    .groupby(["ISO3","country_name","region","wp3_hazard"], as_index=False)
    .agg(
        displacements_sum=("displacements_raw_preferred","sum"),
        record_count=("event_uid","size"),
        unique_event_count=("event_uid","nunique"),
        year_min=("Year","min"),
        year_max=("Year","max"),
    )
    .rename(columns={"ISO3":"iso3","wp3_hazard":"hazard"})
)

agg.to_csv(OUT_AGG, index=False)
print("Wrote aggregates:", OUT_AGG, "rows:", len(agg))
agg.head(10)

Wrote aggregates: C:\pipelines\sewa-multihazar\data\intermediate\idmc_country_hazard_agg_2000_2024.csv rows: 170


,iso3,country_name,region,hazard,displacements_sum,record_count,unique_event_count,year_min,year_max
0,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,7386,1,1,2021,2021
1,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Flash Flooding,15226,15,15,2015,2020
2,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Riverine Flooding Continued & Cluster,551752,65,59,2009,2024
3,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Thunderstorm,96148,69,69,2017,2024
4,BDI,Burundi,East Africa,Drought,9205,4,4,2017,2020
5,BDI,Burundi,East Africa,Flash Flooding,52962,24,24,2018,2020
6,BDI,Burundi,East Africa,Riverine Flooding Continued & Cluster,175372,213,213,2008,2024
7,BDI,Burundi,East Africa,Thunderstorm,32790,242,242,2010,2024
8,BDI,Burundi,East Africa,Wildfires,308,2,2,2017,2017
9,BDI,Burundi,East Africa,Wind,28791,77,77,2017,2022


## Step 7 — Load World Bank population

In [10]:
def find_wdi_zip(pop_dir: Path) -> Path | None:
    if pop_dir.exists():
        zips = sorted(pop_dir.glob("API_SP.POP.TOTL*.zip"))
        if zips:
            return zips[0]
    return None

WDI_ZIP = find_wdi_zip(POP_DIR)
if WDI_ZIP is None and FALLBACK_WDI_ZIP.exists():
    WDI_ZIP = FALLBACK_WDI_ZIP
if WDI_ZIP is None:
    raise FileNotFoundError("Could not find WDI zip for SP.POP.TOTL in data/raw/population or /mnt/data.")

print("Using WDI zip:", WDI_ZIP)

def read_wdi_from_zip(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        candidates = [n for n in z.namelist() if n.lower().endswith(".csv") and "metadata" not in n.lower()]
        if not candidates:
            raise ValueError("No data CSV found inside zip (expected API_*.csv).")

        candidates_sorted = sorted(
            candidates,
            key=lambda n: (0 if Path(n).name.startswith("API_") else 1, len(n))
        )
        target = candidates_sorted[0]

        with z.open(target) as f:
            # WDI bulk CSV typically has 4 header note rows
            df = pd.read_csv(f, skiprows=4)
    return df

wdi = read_wdi_from_zip(WDI_ZIP)

required_cols = {"Country Name","Country Code","Indicator Name","Indicator Code"}
missing = required_cols - set(wdi.columns)
if missing:
    raise ValueError(f"Unexpected WDI columns; missing: {missing}")

# Keep the population indicator
wdi_pop = wdi.loc[wdi["Indicator Code"] == "SP.POP.TOTL"].copy()

# Identify year columns
year_cols = [c for c in wdi_pop.columns if re.fullmatch(r"\d{4}", str(c))]
year_cols_int = sorted([int(c) for c in year_cols])

TARGET_YEAR = 2024
usable_years = [y for y in year_cols_int if y <= TARGET_YEAR]
if not usable_years:
    raise ValueError("No year columns <= 2024 found in WDI file.")

# Convert to numeric
for y in usable_years:
    wdi_pop[str(y)] = pd.to_numeric(wdi_pop[str(y)], errors="coerce")

def pick_latest_leq_target(row):
    for y in reversed(usable_years):
        v = row.get(str(y), np.nan)
        if pd.notna(v):
            return float(v), y
    return np.nan, None

vals = wdi_pop.apply(pick_latest_leq_target, axis=1, result_type="expand")
wdi_pop["pop_ref"] = vals[0]
wdi_pop["pop_ref_year"] = vals[1].astype("Int64")

pop_ref = (
    wdi_pop.rename(columns={"Country Code":"iso3","Country Name":"country_name_wdi"})[
        ["iso3","country_name_wdi","pop_ref","pop_ref_year"]
    ]
    .copy()
)
pop_ref["iso3"] = pop_ref["iso3"].astype(str).str.strip().str.upper()

# Filter to WP3 scope
pop_ref = pop_ref.merge(scope[["iso3"]], on="iso3", how="inner")

OUT_POP_REF = OUT_DIR / "pop_wdi_sp_pop_totl_ref_2024.csv"
pop_ref.to_csv(OUT_POP_REF, index=False)

print("Pop ref rows (WP3 scope):", len(pop_ref))
print("Missing pop_ref:", pop_ref["pop_ref"].isna().sum())
pop_ref.head(10)

Using WDI zip: C:\pipelines\sewa-multihazar\data\raw\population\API_SP.POP.TOTL_DS2_en_csv_v2_34.zip
Pop ref rows (WP3 scope): 46
Missing pop_ref: 0


,iso3,country_name_wdi,pop_ref,pop_ref_year
0,AGO,Angola,37885849.0,2024
1,BDI,Burundi,14047786.0,2024
2,BEN,Benin,14462724.0,2024
3,BFA,Burkina Faso,23548781.0,2024
4,BWA,Botswana,2521139.0,2024
5,CAF,Central African Republic,5330690.0,2024
6,CIV,Cote d'Ivoire,31934230.0,2024
7,CMR,Cameroon,29123744.0,2024
8,COD,"Congo, Dem. Rep.",109276265.0,2024
9,COG,"Congo, Rep.",6332961.0,2024


## Step 8 — Compute per-100k rates (using pop_ref_year=2022)

In [11]:
agg2 = agg.merge(pop_ref[["iso3","pop_ref","pop_ref_year"]], on="iso3", how="left")
agg2["displacements_per100k_pop2024"] = np.where(
    agg2["pop_ref"].notna() & (agg2["pop_ref"] > 0) & agg2["displacements_sum"].notna(),
    (agg2["displacements_sum"] / agg2["pop_ref"]) * 100_000,
    np.nan
)

print("Country-hazard rows:", len(agg2))
print("Unique ISO3 missing population:", agg2.loc[agg2["pop_ref"].isna(), "iso3"].nunique())
agg2.head(10)

Country-hazard rows: 170
Unique ISO3 missing population: 1


,iso3,country_name,region,hazard,displacements_sum,record_count,unique_event_count,year_min,year_max,pop_ref,pop_ref_year,displacements_per100k_pop2024
0,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,7386,1,1,2021,2021,37885849.0,2024,19.495406
1,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Flash Flooding,15226,15,15,2015,2020,37885849.0,2024,40.189148
2,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Riverine Flooding Continued & Cluster,551752,65,59,2009,2024,37885849.0,2024,1456.353796
3,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Thunderstorm,96148,69,69,2017,2024,37885849.0,2024,253.783411
4,BDI,Burundi,East Africa,Drought,9205,4,4,2017,2020,14047786.0,2024,65.526340
5,BDI,Burundi,East Africa,Flash Flooding,52962,24,24,2018,2020,14047786.0,2024,377.013146
6,BDI,Burundi,East Africa,Riverine Flooding Continued & Cluster,175372,213,213,2008,2024,14047786.0,2024,1248.396011
7,BDI,Burundi,East Africa,Thunderstorm,32790,242,242,2010,2024,14047786.0,2024,233.417565
8,BDI,Burundi,East Africa,Wildfires,308,2,2,2017,2017,14047786.0,2024,2.192516
9,BDI,Burundi,East Africa,Wind,28791,77,77,2017,2022,14047786.0,2024,204.950446


## Step 9 — Build master long-format rows (hazard-specific only)

In [13]:
YEAR_OUT = 2024
DIM = "Cascade impacts"

a = agg2.copy()
a["dimension"] = DIM
a["source"] = SOURCE
a["indicator_id"] = "IDMC.DISPLACEMENTS_SUM"
a["indicator_name"] = "IDMC: Disaster internal displacements (sum)"
a["value_raw"] = a["displacements_sum"]
a["unit_raw"] = "count_people"
a["year"] = YEAR_OUT
a["time_window"] = TIME_WINDOW
a["notes"] = (
    f"Sum over IDMC coverage years {TIME_WINDOW} within WP3 framework 2000-2024. "
    "New displacements (flows); may include repeat displacement. "
    "Hazard mapping used hazard fields + Event Codes prefixes + Event Name regex."
)

b = agg2.copy()
b = b.loc[b["displacements_per100k_pop2024"].notna()].copy()
b["dimension"] = DIM
b["source"] = SOURCE
b["indicator_id"] = "IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF"
b["indicator_name"] = "IDMC: Disaster internal displacements (per 100,000; denom WB pop_ref)"
b["value_raw"] = b["displacements_per100k_pop2024"]
b["unit_raw"] = "displacements_per100k"
b["year"] = YEAR_OUT
b["time_window"] = TIME_WINDOW
b["notes"] = (
    "Rate per 100k using World Bank 'Population 2022' table (ISO3-level; values in thousands converted to persons). "
    "Temporary denominator; replace with a 2024 population table when available."
)

long_df = pd.concat([a[LONG_COLUMNS], b[LONG_COLUMNS]], ignore_index=True)
long_df = long_df.loc[long_df["hazard"].isin(HAZARDS_9)].copy()

assert_contract(long_df)
print("Master rows prepared:", len(long_df))
long_df.head(12)

Master rows prepared: 338


,iso3,country_name,region,hazard,dimension,source,indicator_id,indicator_name,value_raw,unit_raw,year,time_window,notes
0,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Drought,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),7386.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...
1,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Flash Flooding,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),15226.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...
2,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Riverine Flooding Continued & Cluster,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),551752.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...
3,AGO,Angola,Southern Africaincl. Indian Ocean Islands Country,Thunderstorm,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),96148.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...
4,BDI,Burundi,East Africa,Drought,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),9205.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...
5,BDI,Burundi,East Africa,Flash Flooding,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),52962.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...
6,BDI,Burundi,East Africa,Riverine Flooding Continued & Cluster,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),175372.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...
7,BDI,Burundi,East Africa,Thunderstorm,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),32790.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...
8,BDI,Burundi,East Africa,Wildfires,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),308.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...
9,BDI,Burundi,East Africa,Wind,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_SUM,IDMC: Disaster internal displacements (sum),28791.0,count_people,2024,2008-2024,Sum over IDMC coverage years 2008-2024 within ...


## Step 10 — Mapping audit (unmapped combos)

In [14]:
unm = idmc.loc[idmc["wp3_hazard"].isna()].copy()
unm["displacements_raw_preferred"] = unm["displacements_raw_preferred"].fillna(0)

audit = (
    unm.groupby(["Hazard Category","Hazard Type","Hazard Sub Type"], as_index=False)
       .agg(
           rows=("ISO3","size"),
           displacements_sum=("displacements_raw_preferred","sum")
       )
       .sort_values(["displacements_sum","rows"], ascending=False)
)

audit.to_csv(OUT_UNMAPPED, index=False)
print("Wrote unmapped audit:", OUT_UNMAPPED, "rows:", len(audit))
audit.head(25)

Wrote unmapped audit: C:\pipelines\sewa-multihazar\data\intermediate\idmc_mapping_audit_unmapped.csv rows: 6


,Hazard Category,Hazard Type,Hazard Sub Type,rows,displacements_sum
2,Geophysical,Volcanic activity,Volcanic activity,2,603348
5,Weather related,Mass Movement,Landslide/Wet mass movement,152,119245
0,Geophysical,Earthquake,Earthquake,13,63886
3,Mixed disasters,Mixed disasters,Mixed disasters,9,13490
1,Geophysical,Mass Movement,Dry mass movement,12,1703
4,Weather related,Erosion,Erosion,1,27


## Step 11 — Upsert into master

In [15]:
master = upsert_to_master(long_df, MASTER_PATH)
print("Updated master:", MASTER_PATH, "rows:", len(master))
master.tail(12)

Updated master: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv rows: 1343


,iso3,country_name,region,hazard,dimension,source,indicator_id,indicator_name,value_raw,unit_raw,year,time_window,notes
1331,ZAF,South Africa,Southern Africaincl. Indian Ocean Islands Country,Wind,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,5.027560,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...
1332,ZMB,Zambia,Southern Africaincl. Indian Ocean Islands Country,Drought,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,713.316040,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...
1333,ZMB,Zambia,Southern Africaincl. Indian Ocean Islands Country,Flash Flooding,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,2.622572,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...
1334,ZMB,Zambia,Southern Africaincl. Indian Ocean Islands Country,Riverine Flooding Continued & Cluster,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,552.813714,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...
1335,ZMB,Zambia,Southern Africaincl. Indian Ocean Islands Country,Thunderstorm,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,17.396236,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...
1336,ZMB,Zambia,Southern Africaincl. Indian Ocean Islands Country,Wind,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,1.205726,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...
1337,ZWE,Zimbabwe,Southern Africaincl. Indian Ocean Islands Country,Drought,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,534.651952,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...
1338,ZWE,Zimbabwe,Southern Africaincl. Indian Ocean Islands Country,Flash Flooding,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,6.023672,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...
1339,ZWE,Zimbabwe,Southern Africaincl. Indian Ocean Islands Country,Riverine Flooding Continued & Cluster,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,442.926223,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...
1340,ZWE,Zimbabwe,Southern Africaincl. Indian Ocean Islands Country,Thunderstorm,Cascade impacts,IDMC (GIDD disasters),IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,IDMC: Disaster internal displacements (per 100...,54.110846,displacements_per100k,2024,2008-2024,Rate per 100k using World Bank 'Population 202...


## Flags
- **IDMC disasters only from 2008**: do not interpret this dataset as “2000–2024”.
- **Flood subtype split** is improved via Event Codes + Event Name, but still imperfect—use `mapping_confidence`.
- **Population year per 100K 2024**: keep the per-100k label explicit until we swap denominators.